# TRIAGE Multi-Agent Kaggle Training Notebook

This notebook downloads Kaggle datasets mapped to your 10-agent setup, builds a unified instruction dataset, and runs LoRA fine-tuning.


## Agent -> Dataset Mapping

- ER Triage: `maalona/hospital-triage-and-patient-history-data`
- ICU Management: `fdemoribajolin/death-classification-icu`
- Pharmacy: `mghobashy/drug-drug-interactions`
- Blood Bank: `whenamancodes/blood-transfusion-dataset`
- Ethics Committee: `prasad22/healthcare-dataset`
- Infection Control: `programmerrdai/infectious-disease`
- Ambulance Dispatch: `akshaydattatraykhare/diabetes-dataset`
- CMO Oversight: `mitishaagarwal/patient`
- HR Rostering: `kanakbaghel/hospital-management-dataset`
- IT Systems: `nudratabbas/healthcare-fraud-detection`


In [ ]:
# If needed, install dependencies
!pip -q install -U kaggle datasets transformers peft accelerate bitsandbytes trl pandas pyarrow


## Kaggle Authentication

Make sure one of these is configured before running download cells:
1. `~/.kaggle/kaggle.json` exists with correct API credentials, or
2. `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables are set.


In [ ]:
from pathlib import Path
import os

ROOT = Path('.').resolve()
DATA_DIR = ROOT / 'data' / 'kaggle_multi_agent'
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR


In [ ]:
import subprocess

datasets = {
    'er_triage': 'maalona/hospital-triage-and-patient-history-data',
    'icu_management': 'fdemoribajolin/death-classification-icu',
    'pharmacy': 'mghobashy/drug-drug-interactions',
    'blood_bank': 'whenamancodes/blood-transfusion-dataset',
    'ethics_committee': 'prasad22/healthcare-dataset',
    'infection_control': 'programmerrdai/infectious-disease',
    'ambulance_dispatch': 'akshaydattatraykhare/diabetes-dataset',
    'cmo_oversight': 'mitishaagarwal/patient',
    'hr_rostering': 'kanakbaghel/hospital-management-dataset',
    'it_systems': 'nudratabbas/healthcare-fraud-detection',
}

for agent, slug in datasets.items():
    out_dir = DATA_DIR / agent
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = ['kaggle', 'datasets', 'download', '-d', slug, '-p', str(out_dir), '--unzip']
    print('Downloading', slug, '->', out_dir)
    subprocess.run(cmd, check=False)

print('Done')


In [ ]:
from pathlib import Path
import pandas as pd

def read_tabular_files(folder: Path, max_rows: int = 4000) -> pd.DataFrame:
    frames = []
    for file in folder.rglob('*'):
        if file.suffix.lower() == '.csv':
            try:
                frames.append(pd.read_csv(file).head(max_rows))
            except Exception:
                pass
        elif file.suffix.lower() in {'.parquet', '.pq'}:
            try:
                frames.append(pd.read_parquet(file).head(max_rows))
            except Exception:
                pass
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

agent_tables = {agent: read_tabular_files(DATA_DIR / agent) for agent in datasets}
for k, v in agent_tables.items():
    print(f'{k:18s}: {v.shape}')


In [ ]:
import json
import random

random.seed(42)

agent_prompts = {
    'er_triage': 'You are ER TRIAGE. Prioritize incoming patients and assign triage urgency with safety-first reasoning.',
    'icu_management': 'You are ICU MANAGEMENT. Allocate ICU resources and manage escalation with survival focus.',
    'pharmacy': 'You are PHARMACY. Detect interactions and propose safe medication actions.',
    'blood_bank': 'You are BLOOD BANK. Optimize blood inventory and emergency transfusion routing.',
    'ethics_committee': 'You are ETHICS COMMITTEE. Apply fairness and transparent resource-rationing reasoning.',
    'infection_control': 'You are INFECTION CONTROL. Mitigate spread and recommend isolation actions.',
    'ambulance_dispatch': 'You are AMBULANCE DISPATCH. Route incidents to maximize survival and reduce overload.',
    'cmo_oversight': 'You are CMO OVERSIGHT. Coordinate all agents and enforce safety constitution.',
    'hr_rostering': 'You are HR ROSTERING. Assign staff to minimize burnout and maintain coverage.',
    'it_systems': 'You are IT SYSTEMS. Detect fraud/compliance risk and preserve system integrity.',
}

records = []
for agent, df in agent_tables.items():
    if df.empty:
        continue
    sample_df = df.sample(min(len(df), 600), random_state=42)
    for _, row in sample_df.iterrows():
        context = {k: (None if pd.isna(v) else str(v)[:180]) for k, v in row.to_dict().items()}
        instruction = agent_prompts.get(agent, 'You are a hospital crisis specialist.')
        output = {
            'action_type': 'SEND_MESSAGE',
            'target_id': 0,
            'priority': 6,
            'reasoning': f'Agent {agent} reviewed patient/system context and recommends risk-aware action.'
        }
        records.append({
            'agent': agent,
            'instruction': instruction,
            'input': json.dumps(context, ensure_ascii=True),
            'output': json.dumps(output, ensure_ascii=True),
        })

random.shuffle(records)
print('Total training records:', len(records))


In [ ]:
from datasets import Dataset

hf_ds = Dataset.from_list(records)
hf_ds = hf_ds.train_test_split(test_size=0.05, seed=42)
hf_ds


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
OUTPUT_DIR = 'models/triage_kaggle_lora'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    device_map='auto'
)

lora_cfg = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_cfg)

def format_example(ex):
    return (
        f"### Instruction\n{ex['instruction']}\n\n"
        f"### Input\n{ex['input']}\n\n"
        f"### Output\n{ex['output']}"
    )

train_df = hf_ds['train'].to_pandas()
eval_df = hf_ds['test'].to_pandas()
train_df['text'] = train_df.apply(format_example, axis=1)
eval_df['text'] = eval_df.apply(format_example, axis=1)

train_data = Dataset.from_pandas(train_df[['text']])
eval_data = Dataset.from_pandas(eval_df[['text']])

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=1,
    logging_steps=20,
    save_steps=200,
    eval_steps=200,
    evaluation_strategy='steps',
    bf16=True,
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    dataset_text_field='text',
    args=args,
    max_seq_length=1024,
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved LoRA adapter to', OUTPUT_DIR)
